# L59 · PyTorch Tensor 基础 — 与 NumPy 互转、device、requires_grad

**目标**：掌握 `torch.Tensor` 的基础操作，理解和 numpy 的异同，为 GPU 训练做准备

🔗 **Aurora 连接**：Month 2 数据集加载器把 `aurora.audio.mfcc.mfcc()` 输出的 numpy MFCC 特征矩阵转成 `torch.Tensor` 送入神经网络；本节的 `mel_to_batch()` 就是那个转换层。

← **上一课**　[L58 · 训练循环](L58_training_loop.ipynb)

> 上节课学习了 **训练循环**：loss 计算、optimizer.step、收敛曲线，拟合 makemoons 数据集。  
> 本课将探讨 **PyTorch Tensor 基础**。

numpy 擅长 CPU 数值计算，但它不知道「梯度（gradient）」是什么，也无法把数据搬到 GPU。
PyTorch 的 `Tensor` 在 ndarray 的基础上增加了两件事：**设备感知**（`.device` 属性决定数据在 CPU 还是 CUDA 核心上）和**计算图追踪**（`requires_grad=True` 时每个算子都被记录，供反向传播（backpropagation）使用）。
把这两点理解透彻，后续所有网络训练代码都只是这两件事的组合。

In [ ]:
try:
    import torch
    import numpy as np
    from aurora.audio.mfcc import mfcc
    HAS_TORCH = True
except ImportError as e:
    HAS_TORCH = False
    print(f'⚠️ {e}；PyTorch 相关 cell 将跳过')


## 1. Tensor vs ndarray：dtype · shape · device

| 属性 | numpy | torch |
|------|-------|-------|
| 类型 | `arr.dtype` | `t.dtype`（`torch.float32` 等）|
| 形状 | `arr.shape` → tuple | `t.shape` → `torch.Size` |
| 设备 | 始终 CPU | `t.device`：`cpu` 或 `cuda:0` |

dtype 默认差异：`np.array([1.0])` 是 `float64`，`torch.tensor([1.0])` 是 `float32`。
神经网络权重通常用 `float32`，混合精度训练（mixed precision training）用 `float16` 或 `bfloat16`。

互转：`torch.from_numpy(arr)` 共享内存（零拷贝），`t.numpy()` 反向转回（仅 CPU Tensor）。

In [ ]:
arr = np.array([[1.0, 2.0], [3.0, 4.0]])   # float64
t   = torch.from_numpy(arr)                 # 共享内存，dtype=float64
t32 = t.float()                             # 转 float32

print('numpy dtype :', arr.dtype)
print('torch dtype :', t.dtype, '->', t32.dtype)
print('shape        :', t.shape)
print('device       :', t.device)

# 修改 arr 会影响 t（共享内存）
arr[0, 0] = 99.0
print('t[0,0] after arr mutation:', t[0, 0].item())  # 99.0

## 2. 自动梯度追踪：`requires_grad` 与计算图

`requires_grad=True` 告诉 PyTorch 对该 Tensor 的所有操作都构建有向无环图（DAG）。
调用 `.backward()` 时，PyTorch 沿 DAG 反向传播，把梯度累积（gradient accumulation）到每个叶子节点的 `.grad`。

手算验证：设 `x = 3.0`，`y = x**2 + 2*x`，则 `dy/dx = 2x + 2 = 8`。
```
y = x**2 + 2*x
dy/dx = 2*x + 2  (在 x=3 时 = 8)
```

注意：`.backward()` 默认只对**标量**调用。对非标量输出需先 `.sum()` 或传入 `gradient` 参数。
叶子节点（用户创建、非运算结果）才会积累 `.grad`；中间节点梯度默认不保留（`retain_graph` 除外）。

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x          # y = 9 + 6 = 15
y.backward()            # dy/dx = 2x+2 = 8

print('y      =', y.item())       # 15.0
print('x.grad =', x.grad.item()) # 8.0  ← 和手算一致

## 3. 常用张量（tensor）操作

**形状变换**
- `view(shape)` / `reshape(shape)`：重排维度，总元素数不变。`view` 要求内存连续，`reshape` 自动处理。
- `squeeze(dim)` / `unsqueeze(dim)`：删除/插入大小为 1 的维度。

**拼接**
- `torch.cat([a, b], dim=0)`：沿已有维度拼接，不新增维度。
- `torch.stack([a, b], dim=0)`：新建一个维度再拼接，要求各 Tensor 形状完全相同。

**乘法**
- `@`（等同 `torch.matmul`）：矩阵乘法，支持批量广播。
- `torch.einsum('bi,ij->bj', x, W)`：爱因斯坦求和（Einstein summation，einsum），写出下标比反复 `transpose` 清晰。
```
einsum 规则：重复的下标代表求和（收缩），箭头右侧是输出下标。
'bi,ij->bj'：对 i 求和 → 等价于 x @ W
```

In [ ]:
# --- view / unsqueeze ---
a = torch.arange(12).float()         # shape (12,)
b = a.view(3, 4)                      # (3, 4)
c = b.unsqueeze(0)                    # (1, 3, 4)
print('view:', b.shape, '  unsqueeze:', c.shape)

# --- cat vs stack ---
x = torch.ones(2, 3)
y2 = torch.zeros(2, 3)
print('cat  dim=0:', torch.cat([x, y2], dim=0).shape)   # (4,3)
print('stack dim=0:', torch.stack([x, y2], dim=0).shape) # (2,2,3)

# --- @ 与 einsum ---
W = torch.randn(3, 5)
inp = torch.randn(4, 3)              # batch=4, in_dim=3
out_mm = inp @ W                     # (4, 5)
out_ein = torch.einsum('bi,ij->bj', inp, W)
print('matmul == einsum:', torch.allclose(out_mm, out_ein))

## 4. ✏️ 实现 `mel_to_batch(mel_list)`

**签名**：`mel_list: list[np.ndarray]`，每个数组 shape `(T, n_mels)` → 返回 `torch.Tensor` shape `(B, 1, T, n_mels)`

**推理路线**：
1. `np.stack(mel_list, axis=0)` 把列表堆成 `(B, T, n_mels)`，所有帧数 T 必须相同（调用方负责对齐）
2. `torch.from_numpy(...).float()` 转为 `float32` Tensor（网络权重默认 float32）
3. `.unsqueeze(1)` 在第 1 维插入 channel 轴，得到 `(B, 1, T, n_mels)`（CNN 期望 `B,C,H,W`）

**参考输入输出**：
```
mel_list = [np.zeros((80, 40))] * 3
mel_to_batch(mel_list).shape  # → torch.Size([3, 1, 80, 40])
```

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


In [ ]:
def mel_to_batch(mel_list):
    """
    mel_list : list of np.ndarray, each shape (T, n_mels)
    returns  : torch.Tensor shape (B, 1, T, n_mels), dtype float32
    """
    # ✏️ TODO: 用 np.stack 堆叠列表
    raise NotImplementedError("TODO: 用 np.stack(mel_list, axis=0) 堆叠列表，得到 (B, T, n_mels)")
    # ✏️ TODO: 转为 float32 Tensor
    raise NotImplementedError("TODO: 用 torch.from_numpy(arr).float() 转为 float32 Tensor")
    # ✏️ TODO: unsqueeze channel 维度
    raise NotImplementedError("TODO: 用 t.unsqueeze(1) 插入 channel 轴，返回 (B, 1, T, n_mels)")

In [ ]:
# 检查 mel_to_batch
batch = mel_to_batch([np.zeros((80, 40)) for _ in range(3)])
assert batch.shape == (3, 1, 80, 40), f'shape 错误：{batch.shape}'
assert batch.dtype == torch.float32,   f'dtype 错误：{batch.dtype}'
print('✅ mel_to_batch shape:', batch.shape, '  dtype:', batch.dtype)

## 5. 参数实验：`requires_grad` 与梯度流

实验参数与预期现象：
- `requires_grad=False`（默认）：`.grad` 保持 `None`，调用 `.backward()` 会抛 `RuntimeError`
- `requires_grad=True`：每次 `.backward()` 后 `.grad` 累加（多次调用前需 `x.grad.zero_()`）
- `torch.no_grad()` 上下文：内部操作不建计算图，推理阶段节省内存

下面验证梯度累加陷阱和 `no_grad` 的效果。

In [ ]:
# 实验 A：requires_grad=False 不会有梯度
x_ng = torch.tensor(3.0)           # requires_grad 默认 False
y_ng = x_ng ** 2
print('requires_grad=False, x_ng.grad:', x_ng.grad)  # None

# 实验 B：梯度累加陷阱
x2 = torch.tensor(3.0, requires_grad=True)
for _ in range(3):
    y2 = x2 ** 2 + 2 * x2
    y2.backward()
    # 没有 zero_() 则梯度累加
print('3次 backward 后 x2.grad:', x2.grad.item())  # 8*3 = 24（累加）

# 正确做法：每次迭代清零
x3 = torch.tensor(3.0, requires_grad=True)
for _ in range(3):
    if x3.grad is not None:
        x3.grad.zero_()
    y3 = x3 ** 2 + 2 * x3
    y3.backward()
print('清零后每次 x3.grad:', x3.grad.item())        # 8.0（正确）

# 实验 C：torch.no_grad() 不建图
x4 = torch.tensor(3.0, requires_grad=True)
with torch.no_grad():
    y4 = x4 ** 2
print('no_grad 下 y4.requires_grad:', y4.requires_grad)  # False

## 本课收束

`mel_to_batch()` 把 numpy MFCC 列表转为 `(B, 1, T, n_mels)` 的 `float32` Tensor，是 Aurora Month 2 数据集加载器与 CNN 声学模型之间的桥梁。
`torch.from_numpy` 实现零拷贝转换，`unsqueeze(1)` 插入 channel 维以满足 `Conv2d` 对 `B,C,H,W` 的要求。
`requires_grad` 机制让 Tensor 在前向传播（forward pass）时同步构建计算图，`backward()` 一键完成反向求导。
下一节（L60）将剖析 PyTorch autograd 机制：追踪 grad_fn、手动调用 backward()，理解为何 retain_graph 在有向无环计算图中不可缺少。

---

→ **下一课**　[L60 · autograd 机制](L60_pytorch_autograd.ipynb)

> 下节课将学习 **autograd 机制**：gradfn 计算图，backward()，retaingraph 原理。